
# Adaptive, Information-Weighted Reddit Scraper — **Operational Reference (Standalone Notebook)**

> **Purpose:** This notebook is a **hands-on hand-off** for engineers/agents operating an adaptive Reddit scraper.  
> It is **self-contained** and covers: data hygiene (merge integrity), information-gain modeling, a **continuous** scheduling policy, **bursty rate-limit** control, subreddit-adaptive priors, and controlled lateness for broader coverage.

**Quick wins you get by adopting this design:**
- Dramatically lower waste (near-zero “no change” scrapes with sparse histories)
- Stable throughput under bursty rate limits (600/min nominal; stalls & bursts tolerated)
- Higher total information yield via **continuous priority** + **controlled lateness**
- Clear metrics & logs so you can tune without guesswork



## 1. Core Concepts (What we optimize and why)

### 1.1 Information Gain (ΔInfo)
Operational definition per scrape *t* for a post:
\[
\Delta I_t = \text{new\_comments}_t + \text{edited\_comments}_t + \text{score\_changed}_t + \text{deleted\_comments}_t
\]
Optional weights (recommended defaults):
- `w_new=1.0`, `w_edit=0.5`, `w_score=0.2`, `w_delete=0.1`

### 1.2 Lifecycle Metrics
- **Half-life** (*t_half*): time to reach 50% of cumulative ΔInfo
- **Dormancy**: first occurrence of **3 consecutive** zero-Δ scrapes
- **Waste rate**: fraction of zero-Δ scrapes

### 1.3 Continuous Temperature (T)
We replace discrete states (Hot/Warm/Cool) with a **continuous scalar**:
\[
T_i = \alpha \cdot \frac{\dot{I}_i}{\max_j \dot{I}_j} \, + \, \beta \cdot e^{-\lambda_i h_i} \, + \, \gamma \cdot \frac{\sigma_i}{\max_j \sigma_j}
\]
- \(\dot{I}_i\): recent ΔInfo rate (mean per hour over last k scrapes, k≈3–5)  
- \(h_i\): hours since last non-zero ΔInfo  
- \(\sigma_i\): volatility (std of ΔInfo over last k scrapes)  
- Typical weights: **α=0.6, β=0.3, γ=0.1**



## 2. Data Hygiene (Merge Integrity Must-Haves)

**Sparse histories** and **real scrape timestamps** are required for truthful ΔInfo.

**Rules:**
1. `"No scrape → no entry"` for comment `score_history` (never insert zeros).
2. Use **triple fallback** for timestamps (metadata → filename → mtime), normalize to UTC.
3. **Deletion threshold**: mark `deleted=True` only after 3 consecutive absences (with a one-time audit marker `[None, ts]`).
4. Root `post.score_history` must have **one entry per raw scrape**.

**Integrity assertions (fast checks to run per merge):**
- `len(post.score_history) == len(raw_scrapes)`
- `min_ts` / `max_ts` align with first/last scrape
- No backward time jumps


## 3. Computing ΔInfo and Temperature (T)

In [ ]:

from dataclasses import dataclass, field
from typing import List, Dict, Optional
import math, numpy as np
from datetime import datetime, timezone

@dataclass
class ScrapeDelta:
    ts: datetime
    new_comments: int = 0
    edited_comments: int = 0
    score_changed: int = 0
    deleted_comments: int = 0

    def weighted(self, w_new=1.0, w_edit=0.5, w_score=0.2, w_delete=0.1) -> float:
        return (w_new * self.new_comments
              + w_edit * self.edited_comments
              + w_score * self.score_changed
              + w_delete * self.deleted_comments)

@dataclass
class PostState:
    post_id: str
    deltas: List[ScrapeDelta] = field(default_factory=list)
    last_scrape_ts: Optional[datetime] = None
    last_nonzero_ts: Optional[datetime] = None

    # cache fields (optional):
    recent_rate: float = 0.0        # ΔInfo/hour over last k
    volatility: float = 0.0         # std over last k
    temperature: float = 0.0        # T in [0,1]
    lambda_hat: float = 0.15        # subreddit prior, overridable

def hours_between(a: datetime, b: datetime) -> float:
    return max(0.0, (a - b).total_seconds()/3600.0)

def compute_recent_stats(post: PostState, k: int = 5, weights=(1.0,0.5,0.2,0.1)):
    if not post.deltas:
        post.recent_rate, post.volatility = 0.0, 0.0
        return

    # sort by ts
    ds = sorted(post.deltas, key=lambda d: d.ts)[-k:]
    vals = [d.weighted(*weights) for d in ds]
    if len(ds) >= 2:
        hours = hours_between(ds[-1].ts, ds[0].ts) or 1e-6
        rate = (sum(vals)) / hours
    else:
        rate = vals[-1]  # degenerate window

    post.recent_rate = rate
    post.volatility = float(np.std(vals)) if len(vals) > 1 else 0.0
    # update last_nonzero_ts
    for d in reversed(ds):
        if d.weighted(*weights) > 0:
            post.last_nonzero_ts = d.ts
            break

def compute_temperature(posts: List[PostState], now: datetime, alpha=0.6, beta=0.3, gamma=0.1):
    # normalize recent_rate and volatility across cohort
    max_rate = max((p.recent_rate for p in posts), default=1.0) or 1.0
    max_vol  = max((p.volatility  for p in posts), default=1.0) or 1.0

    for p in posts:
        h = hours_between(now, p.last_nonzero_ts or p.last_scrape_ts or now)
        # three-term temperature
        t = alpha * (p.recent_rate / max_rate)           + beta  * math.exp(-p.lambda_hat * h)           + gamma * (p.volatility / max_vol)
        # clamp to [0,1]
        p.temperature = max(0.0, min(1.0, t))



## 4. Continuous Scheduling, Priority, and Controlled Lateness

### 4.1 Interval function (continuous; differentiable)
\[
\Delta t_i = \mathrm{clamp}\!\left(\frac{k}{T_i + \epsilon},\; 0.5\,\text{h},\; 24\,\text{h}\right)
\]

### 4.2 Lateness-aware priority (run slightly late on purpose)
Define lateness fraction \(\phi_i = (\text{now} - \text{due}_i)/\Delta t_i\).  
Priority:
\[
S_i = \frac{T_i}{1 + \lambda_i \, \phi_i \, \Delta t_i}
\]
Delay low-\(\lambda\) or cool posts first; serve hot/fast-decay posts promptly.

### 4.3 Global lateness setpoint
Maintain avg lateness \(L\) near \(L^*\) (e.g., 10–20%) with slow feedback.


In [ ]:

from dataclasses import dataclass
import heapq, math
from datetime import timedelta

@dataclass(order=True)
class QueueEntry:
    next_time: datetime
    post_id: str

def compute_interval_hours(T: float, k: float = 1.5, g: float = 1.0,
                           min_h: float = 0.5, max_h: float = 24.0) -> float:
    Δt = k / (T + 1e-3)
    Δt = max(min_h, min(max_h, Δt))
    return Δt * g

def compute_priority(T: float, lam: float, lateness_frac: float, Δt: float) -> float:
    return T / (1.0 + lam * lateness_frac * Δt)

def schedule_next(posts: List[PostState], now: datetime, g: float,
                  lateness_setpoint: float = 0.15,
                  clamp_early_lateness: float = 0.25) -> List[QueueEntry]:
    q = []
    for p in posts:
        Δt = compute_interval_hours(p.temperature, g=g)
        due = (p.last_scrape_ts or now) + timedelta(hours=Δt)
        φ = max(0.0, (now - due).total_seconds()/3600.0 / Δt)  # lateness fraction
        # early window clamp (optional): if recently active, limit lateness
        if p.last_nonzero_ts and (now - p.last_nonzero_ts).total_seconds()/3600.0 < Δt:
            φ = min(φ, clamp_early_lateness)
        S = compute_priority(p.temperature, p.lambda_hat, φ, Δt)
        # use due time as primary ordering; priority S as tie-breaker via small epsilon
        eps = max(0.0, 1.0 - S) * 1e-6
        heapq.heappush(q, QueueEntry(next_time=due + timedelta(seconds=eps), post_id=p.post_id))
    return q



## 5. Rate-Limit Control (Bursty) — EMA + Gain Feedback

We target ~**600 requests/min** (\~10 rps), but actual throughput is **bursty**.  
We update a global gain **g** slowly using EMA-smoothed rate error (and optionally queue pressure) so intervals stretch or compress gradually, avoiding oscillations.


In [ ]:

class RateController:
    def __init__(self, target_rps=10.0, ema_alpha=0.03, eta=0.05, g_min=0.5, g_max=3.0):
        self.target = target_rps
        self.alpha = ema_alpha
        self.eta = eta
        self.g = 1.0
        self.g_min = g_min
        self.g_max = g_max
        self.rate_ema = target_rps

    def update(self, instantaneous_rps: float, queue_lateness: float = None, lateness_setpoint: float = None, eta_l=0.02):
        # Update EMA of achieved rate
        self.rate_ema = self.alpha * instantaneous_rps + (1 - self.alpha) * self.rate_ema
        # Rate error
        e_r = (self.rate_ema / self.target) - 1.0
        d = self.eta * e_r

        # Optional lateness pressure term to run "slightly late on purpose"
        if queue_lateness is not None and lateness_setpoint is not None:
            d += eta_l * (queue_lateness - lateness_setpoint)

        # Exponential update, clipped for stability
        self.g *= math.exp(d)
        self.g = max(self.g_min, min(self.g_max, self.g))
        return self.g



## 6. Subreddit-Adaptive Priors

Maintain rolling priors per subreddit:
- \(\lambda\) (decay)
- burstiness (std/mean ΔInfo)
- depth ratio, edit rate, deletion fraction, diurnal pattern

New posts inherit priors immediately; posts update priors at dormancy/retire.


In [ ]:

from collections import defaultdict, deque

class SubredditPriors:
    def __init__(self, max_history=200):
        self.hist = defaultdict(lambda: deque(maxlen=max_history))
        self.lambda_prior = defaultdict(lambda: 0.15)  # default

    def record_post(self, subreddit: str, lambda_est: float):
        self.hist[subreddit].append(lambda_est)
        # simple rolling mean (could be EWMA)
        vals = list(self.hist[subreddit])
        self.lambda_prior[subreddit] = float(sum(vals)/len(vals))

    def get_lambda(self, subreddit: str) -> float:
        return self.lambda_prior[subreddit]



## 7. Metrics & Observability (what to log, alert, and tune)

**Per scrape cycle (rolling):**
- Throughput: instantaneous rps, EMA rps  
- Global gain `g`  
- Average lateness vs. setpoint (e.g., 15%)  
- Queue size (ready posts), boundedness  
- Info gain/request (efficiency)  
- Coverage: fraction of active posts touched in last 6–12h

**Per post (telemetry fields):**
- T (temperature), Δt (interval), λ (decay prior)  
- Hours since change, lateness fraction φ  
- State transitions (if you still emit them for debug): Hot→Warm→Dormant



## 8. End-to-End Control Loop (pseudo-structured code)
Integrate with your fetcher/merger. This sketch focuses on scheduling and control.


In [ ]:

from datetime import datetime, timezone, timedelta

def control_tick(posts: List[PostState],
                 priors: SubredditPriors,
                 controller: RateController,
                 now: datetime,
                 lateness_setpoint=0.15):
    # 1) Update per-post stats & temperature
    for p in posts:
        compute_recent_stats(p, k=5)
    compute_temperature(posts, now)

    # 2) Build time-ordered queue
    q = schedule_next(posts, now, g=controller.g, lateness_setpoint=lateness_setpoint)

    # 3) Decide how many to scrape this tick (driven by external rate window)
    # Example: assume we allow up to N requests this second based on external feedback
    # Here we simulate 'ready' and compute a dummy instantaneous rate:
    ready_now = [e for e in q if e.next_time <= now]
    instantaneous_rps = float(len(ready_now))  # replace with actual completions

    # Average lateness among ready posts (for optional feedback)
    if ready_now:
        latenesses = []
        for p in posts:
            Δt = compute_interval_hours(p.temperature, g=controller.g)
            due = (p.last_scrape_ts or now) + timedelta(hours=Δt)
            φ = max(0.0, (now - due).total_seconds()/3600.0 / Δt)
            latenesses.append(φ)
        avg_lateness = sum(latenesses)/len(latenesses)
    else:
        avg_lateness = 0.0

    # 4) Update global gain slowly using EMA + lateness pressure
    controller.update(instantaneous_rps, avg_lateness, lateness_setpoint)

    # 5) Return queue (the agent decides how many to pop based on actual credits)
    return q



## 9. Optional: Burst/Stall Simulation (sanity test)
This cell simulates 50 posts with different λ over 30 minutes under bursty throughput and shows the controller’s stability.


In [ ]:

import numpy as np, random, math
import matplotlib.pyplot as plt

def run_simulation(sim_minutes=30, n_posts=50, lambda_mean=0.15,
                   burst_prob=0.05, stall_prob=0.03, target_rps=10.0):
    # init posts
    rng = np.random.default_rng(42)
    lambdas = rng.lognormal(mean=np.log(lambda_mean), sigma=0.5, size=n_posts)
    posts = [PostState(post_id=f"p{i}", last_scrape_ts=datetime.now(timezone.utc),
                       last_nonzero_ts=datetime.now(timezone.utc), lambda_hat=float(lambdas[i]))
             for i in range(n_posts)]
    # seed tiny deltas to initialize stats
    now = datetime.now(timezone.utc)
    for p in posts:
        p.deltas.append(ScrapeDelta(ts=now, new_comments=1))

    priors = SubredditPriors()
    ctrl = RateController(target_rps=target_rps)

    SECONDS = int(sim_minutes*60)
    rate_hist, ema_hist, gain_hist, info_hist = [], [], [], []
    burst_timer, stall_timer = 0, 0
    total_requests = 0
    info_yield = 0.0

    for s in range(SECONDS):
        now = datetime.now(timezone.utc) + timedelta(seconds=s)
        # decay proxy: reduce synthetic ΔInfo tendency
        for p in posts:
            # simple synthetic decay hits temperature indirectly via stats
            pass

        # build queue and compute instantaneous capacity
        q = control_tick(posts, priors, ctrl, now)
        # simulate burst/stall
        if burst_timer == 0 and stall_timer == 0:
            if random.random() < burst_prob:
                burst_timer = random.randint(3, 10)
            elif random.random() < stall_prob:
                stall_timer = random.randint(30, 240)

        if stall_timer > 0:
            R_t = 0
            stall_timer -= 1
        elif burst_timer > 0:
            R_t = min(len([e for e in q if e.next_time <= now]), random.randint(20, 40))
            burst_timer -= 1
        else:
            R_t = min(len([e for e in q if e.next_time <= now]), int(target_rps + np.random.randn()*2))

        # execute R_t scrapes: add synthetic ΔInfo to keep stats moving
        if R_t > 0:
            total_requests += R_t
            info_gain = 0.0
            # randomly pick R_t posts that are ready
            ready = [e for e in q if e.next_time <= now]
            ready_ids = [e.post_id for e in ready]
            chosen_ids = set(rng.choice(ready_ids, size=min(len(ready_ids), R_t), replace=False)) if ready_ids else set()
            # add synthetic deltas proportional to a random temperature-like proxy
            for p in posts:
                if p.post_id in chosen_ids:
                    # synthetic delta: emulate decaying activity
                    val = rng.uniform(0.2, 1.0)
                    p.deltas.append(ScrapeDelta(ts=now, new_comments=int(val > 0.6),
                                                edited_comments=int(0.3 < val < 0.6),
                                                score_changed=int(val > 0.4)))
                    p.last_scrape_ts = now
                    if val > 0.4:
                        p.last_nonzero_ts = now
                    info_gain += val
            info_yield += info_gain

        # record
        rate_hist.append(R_t)
        ema_hist.append(ctrl.rate_ema)
        gain_hist.append(ctrl.g)
        info_hist.append(info_yield)

    # Plot
    t_min = np.arange(len(rate_hist))/60.0
    fig, ax1 = plt.subplots(figsize=(11,5))
    ax1.plot(t_min, rate_hist, color='grey', alpha=0.35, label='Instant req/s')
    ax1.plot(t_min, ema_hist, color='blue', label='EMA req/s')
    ax1.axhline(target_rps, color='red', linestyle='--', label='Target')
    ax1.set_xlabel('Minutes'); ax1.set_ylabel('Req/s'); ax1.legend(loc='upper left')

    ax2 = ax1.twinx()
    ax2.plot(t_min, gain_hist, color='green', label='Gain g', alpha=0.8)
    ax2.plot(t_min, info_hist, color='purple', label='Cumulative ΔInfo', alpha=0.7)
    ax2.set_ylabel('Gain / Cumulative ΔInfo'); ax2.legend(loc='upper right')
    plt.title('Bursty Rate-Limit Control — Stability Check')
    plt.tight_layout()
    plt.show()

run_simulation(sim_minutes=10)  # quick sanity run; change to 30 for a longer view



## 10. Tuning & Defaults (copy/paste friendly)

- Temperature weights: `α=0.6, β=0.3, γ=0.1`  
- Interval: `Δt = clamp(k/(T+1e-3), 0.5h, 24h)` with **k=1.5** (start here)  
- Rate controller: `EMA α=0.03`, `η=0.05`, clamps `g∈[0.5, 3.0]`  
- Lateness setpoint: **L\*=0.15** (15%)  
- Early-window lateness clamp: **25%**  
- Dormancy detection: **3** consecutive zero-Δ scrapes  
- Deletion threshold: **3** consecutive absences + audit marker `[None, ts]`
